# Small MoE LLM — train on Colab GPU

Runs the full pipeline (data prep → train → evaluate → ablations) on a Colab GPU in **bfloat16**.

**Before running:** set the runtime to GPU — *Runtime → Change runtime type → T4 GPU (or A100)*.

Fill in `REPO_URL` below with your GitHub repo, then Run All.

In [ ]:
# 0. Check the GPU
!nvidia-smi

In [ ]:
# 1. Clone the repo (edit REPO_URL to your fork)
REPO_URL = 'https://github.com/iamrinni/Small_MoE_LLM.git'
!git clone $REPO_URL
%cd Small_MoE_LLM

In [ ]:
# 2. Install deps. Colab already ships a CUDA build of torch — do NOT reinstall it
#    (that could downgrade to CPU). Install everything else.
!grep -v '^torch' requirements.txt > /tmp/reqs_colab.txt
!pip install -q -r /tmp/reqs_colab.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| bf16', torch.cuda.is_bf16_supported())

In [ ]:
# 3. Sanity: run the test suite (forced onto CPU by conftest) + a tiny smoke train
!python -m pytest -q tests/test_model_build.py tests/test_trainer.py
!python scripts/train.py --config configs/smoke.yaml

In [ ]:
# 4. Prepare real data (streams a capped subsample of each modality)
!python scripts/prepare_data.py --config configs/train_small.yaml --set data.max_samples_per_modality=20000

In [ ]:
# 5. Train the FULL model (172M, bf16, real data).
#    T4 (15GB): small batch + shorter seq + grad accumulation + gradient checkpointing.
#    save_total_limit keeps only the last 2 step_* checkpoints so the disk doesn't fill up.
#    On an A100 you can raise batch to 8-16, seq to 1024, and drop gradient_checkpointing.
!python scripts/train.py --config configs/train_small.yaml --set \
    training.max_steps=3000 \
    training.per_device_batch_size=2 \
    training.grad_accum_steps=16 \
    training.gradient_checkpointing=true \
    data.max_seq_len=512 \
    eval.save_total_limit=2 \
    logging.backend=tensorboard

In [ ]:
# 6. Watch training curves in TensorBoard
%load_ext tensorboard
%tensorboard --logdir checkpoints/small-moe-baseline/tb

In [ ]:
# 7. Evaluate the trained checkpoint (metrics + routing heatmap)
!python scripts/evaluate.py --config configs/train_small.yaml \
    --checkpoint checkpoints/small-moe-baseline/final --n_examples 200
from IPython.display import Image
Image('checkpoints/small-moe-baseline/final/routing_heatmap.png')

In [ ]:
# 8. (optional) Run the ablation matrix on GPU
!python scripts/run_ablations.py --steps 500 --out report/figures
from IPython.display import Image
Image('report/figures/ablation_comparison.png')

In [ ]:
# 9. Plot training metrics, then download EVERYTHING to your PC.
#    Colab's disk is wiped when the runtime disconnects — download before you leave.
!python scripts/plot_metrics.py --log logs/metrics.jsonl --out report/figures

CKPT = 'checkpoints/small-moe-baseline'
# bundle: trained model weights + config, logs (JSONL + TensorBoard),
# eval metrics, ablation stats, and all figures.
!rm -f moe_results.zip
!zip -r moe_results.zip \
    {CKPT}/final/hf \
    {CKPT}/final/results.json \
    {CKPT}/final/summary.md \
    {CKPT}/final/routing_heatmap.png \
    {CKPT}/tb \
    logs \
    report/figures \
    2>/dev/null

!echo "archive size:" && du -h moe_results.zip
from google.colab import files
files.download('moe_results.zip')   # saves to your PC's Downloads folder